# Futures/Spot HBT Daily-Pair Full-Market Visual Backtest

本 notebook 是可視化回測 runner。  
調用 scripts：每個 code cell 都列出實際調用的 script；只保留 notebook 必要的 runner / visualization code。

本 notebook 在 `future_spot/` 底下做 2026-05-21 到 2026-05-26 的全市場股票期貨 / 現貨 paired HBT 回測。Pair universe 不是直接拿固定 config 的 pairs，而是用 `future_spot/scripts/build_arbitrage_config_from_date.py` 依每個交易日產生 daily pairs，再逐日回測。

流程：
1. 用 `future_spot/scripts/build_arbitrage_config_from_date.py` 產生每日 arbitrage config 與每日 pairs。
2. 用 `scripts/tw_stock_data_to_npz.py` 產生 spot / future 的 HBT event `.npz`。
3. 用 `scripts/tw_stock_hftbacktest.py` 的 `BacktestConfig` / `event_summary` 稽核 HBT 設定。
4. 在 notebook 內定義進場信號，輸出每個 daily pair 的 signal DataFrame。
5. 用 `future_spot/arbitrage/hbt_backtest.py` 執行 paired HBT，最後做 summary、entry/exit、latency 視覺化。


In [1]:
# 本 cell：設定 repo 路徑、匯入 notebook 實際需要的套件與 scripts。
# 調用 scripts:
# - scripts/tw_stock_data_to_npz.py: convert_tw_stock_to_npz, convert_tw_stock_future_to_npz, default_output_path
# - scripts/tw_stock_hftbacktest.py: BacktestConfig, event_summary
# - future_spot/scripts/build_arbitrage_config_from_date.py: build_arbitrage_config_from_date
# - future_spot/arbitrage/config.py: load_config
# - future_spot/arbitrage/hbt_backtest.py: HbtPairBacktester and HBT pair configs

from __future__ import annotations

from dataclasses import replace
from pathlib import Path
import sys
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%matplotlib inline


def find_future_spot_root() -> Path:
    start = Path.cwd().resolve()
    for path in (start, *start.parents):
        if path.name == "future_spot" and (path / "arbitrage").is_dir():
            return path
        candidate = path / "future_spot"
        if (candidate / "arbitrage").is_dir() and (path / "scripts").is_dir():
            return candidate
    raise FileNotFoundError(f"cannot find future_spot root from {start}")


FUTURE_SPOT_ROOT = find_future_spot_root()
FUTURE_SPOT_SCRIPT_ROOT = FUTURE_SPOT_ROOT / "scripts"
ROOT = FUTURE_SPOT_ROOT.parent
for import_path in (FUTURE_SPOT_SCRIPT_ROOT, FUTURE_SPOT_ROOT, ROOT):
    import_text = str(import_path)
    if import_text not in sys.path:
        sys.path.insert(0, import_text)

from scripts.tw_stock_data_to_npz import (
    convert_tw_stock_future_to_npz,
    convert_tw_stock_to_npz,
    default_output_path,
)
from scripts.tw_stock_hftbacktest import BacktestConfig, event_summary
from build_arbitrage_config_from_date import build_arbitrage_config_from_date
from arbitrage.config import load_config
from arbitrage.hbt_backtest import (
    HbtAssetConfig,
    HbtPairBacktestConfig,
    HbtPairBacktester,
    infer_hbt_asset_tick_size,
)
from arbitrage.models import PairConfig, Signal

print(f"ROOT={ROOT}")
print(f"FUTURE_SPOT_ROOT={FUTURE_SPOT_ROOT}")
print(f"FUTURE_SPOT_SCRIPT_ROOT={FUTURE_SPOT_SCRIPT_ROOT}")


ROOT=C:\Users\zoufuc\Desktop\hftbacktest
FUTURE_SPOT_ROOT=C:\Users\zoufuc\Desktop\hftbacktest\future_spot
FUTURE_SPOT_SCRIPT_ROOT=C:\Users\zoufuc\Desktop\hftbacktest\future_spot\scripts


## Parameters

本 cell 設定 2026-05-21 到 2026-05-26 的每日 pair 建構、HBT data conversion 與 latency 參數。  
調用 scripts：本段只設定參數，下一個 code cell 才會呼叫 `future_spot/scripts/build_arbitrage_config_from_date.py`。


In [2]:
# 本 cell：設定資料日期、每日 pair 建構參數、HBT 延遲與輸出路徑。
# 調用 scripts：本 cell 不直接調用 script；參數會傳給後續 daily config builder、conversion 與 HBT runner。

BASE_CONFIG_FILE = FUTURE_SPOT_ROOT / "arbitrage_config_20260702.json"
CALENDAR_FILE = FUTURE_SPOT_ROOT / "Calendar.csv"
STOCKINFO_FILE = FUTURE_SPOT_ROOT / "stockinfo.csv"

TRADE_START_DATE = "2026-05-21"
TRADE_END_DATE = "2026-05-26"
SESSION_START_TIME = "09:00:00"
SESSION_END_TIME = "13:30:00"

# 每日 pair 產生用的 target 篩選參數；None 表示使用 builder script 的預設值。
REBUILD_DAILY_CONFIGS = True
RAISE_ON_DAILY_CONFIG_ERROR = True
BUILD_SESSION_START_TIME = "08:45:00"
BUILD_SESSION_END_TIME = "13:45:00"
MIN_FUTURE_VOLUME = 1000
MIN_STOCK_VOLUME = 20_000_000
REQUIRED_UNIT = 2000
NAME_TEMPLATE = "{spot_symbol}_{future_symbol}"
FUTURES_PARQUET_TEMPLATE = r"\\DC_TW\taiwan_stock\ticks_parquet_stock_future\{ldate}.parquet"
TWSE_DAYTRADE_TEMPLATE: str | None = None
TPEX_DAYTRADE_TEMPLATE: str | None = None
TWSE_DAILY_TEMPLATE: str | None = None
TPEX_DAILY_TEMPLATE: str | None = None

# 全市場預設使用每日 config 產生的所有 pairs；需要 smoke test 時可填 pair names，例如 ["1301_CFFG6"]。
PAIR_NAME_FILTER: list[str] = []
MAX_PAIRS: int | None = None

# 先生成 HBT event npz；若資料已存在且 REBUILD_EVENT_DATA=False，會直接重用。
CONVERT_MISSING_EVENT_DATA = True
REBUILD_EVENT_DATA = False
CONVERSION_QA_SAMPLE_ROWS = 1000

# HBT execution settings；後續 HBT Settings Audit 會用 BacktestConfig 明確列出設定。
FIRST_LEG = "future"
STEP_MS = 1000.0
ORDER_LATENCY_MS = 0.0
RESPONSE_LATENCY_MS = 0.0
FEED_LATENCY_OFFSET_MS = 0.0
SECOND_LEG_DELAY_MS = 0.0
RESPONSE_TIMEOUT_MS = 50.0
MAX_STEPS: int | None = None
MAX_TRADES_PER_PAIR: int | None = None
RECORD_MARKET_EVERY_STEPS = 1
QUEUE_MODEL = "risk_adverse"
SECOND_LEG_PROFIT_CHECK = True
FLATTEN_ON_SECOND_LEG_FAILURE = True
PAIR_OVERRIDES: dict[str, float] = {}

OUTPUT_DIR = FUTURE_SPOT_ROOT / "output" / "hbt_pair_full_market_20260521_20260526"
DAILY_CONFIG_DIR = OUTPUT_DIR / "daily_configs"
DAILY_TARGET_DIR = OUTPUT_DIR / "daily_targets"
EVENT_OUTPUT_DIR = ROOT / "data"
for path in (OUTPUT_DIR, DAILY_CONFIG_DIR, DAILY_TARGET_DIR, EVENT_OUTPUT_DIR):
    path.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 200)


## Build Daily Pair Universe

本 cell 用 `build_arbitrage_config_from_date.py` 逐交易日建立 daily config，再讀回每天的 pair universe。  
調用 scripts：`future_spot/scripts/build_arbitrage_config_from_date.py`、`future_spot/arbitrage/config.py` 的 `load_config`。


In [4]:
# 本 cell：逐日調用 build_arbitrage_config_from_date.py，建立 daily pair universe。
# 調用 scripts:
# - future_spot/scripts/build_arbitrage_config_from_date.py: build_arbitrage_config_from_date
# - future_spot/arbitrage/config.py: load_config


def normalize_trade_date(value: object) -> str:
    return pd.Timestamp(value).strftime("%Y-%m-%d")


def select_calendar_trade_dates(calendar_file: Path, start_date: str, end_date: str) -> list[str]:
    calendar = pd.read_csv(calendar_file, dtype=str)
    start = normalize_trade_date(start_date)
    end = normalize_trade_date(end_date)
    dates = [normalize_trade_date(value) for value in calendar["trade_dates"].dropna().astype(str)]
    selected = [value for value in dates if start <= value <= end]
    if not selected:
        raise ValueError(f"No trade dates found between {start} and {end} in {calendar_file}")
    return selected


def pair_run_key(trade_date: str, pair_name: str) -> str:
    return f"{trade_date}::{pair_name}"


def daily_builder_kwargs(trade_date: str) -> dict[str, Any]:
    date_nodash = trade_date.replace("-", "")
    kwargs: dict[str, Any] = {
        "base_config": BASE_CONFIG_FILE,
        "calendar": CALENDAR_FILE,
        "stockinfo": STOCKINFO_FILE,
        "session_start": BUILD_SESSION_START_TIME,
        "session_end": BUILD_SESSION_END_TIME,
        "min_future_volume": MIN_FUTURE_VOLUME,
        "min_stock_volume": MIN_STOCK_VOLUME,
        "required_unit": REQUIRED_UNIT,
        "output": DAILY_CONFIG_DIR / f"arbitrage_config_{date_nodash}.json",
        "target_output": DAILY_TARGET_DIR / f"target_futures_{date_nodash}.csv",
        "name_template": NAME_TEMPLATE,
    }
    optional_values = {
        "futures_parquet_template": FUTURES_PARQUET_TEMPLATE,
        "twse_daytrade_template": TWSE_DAYTRADE_TEMPLATE,
        "tpex_daytrade_template": TPEX_DAYTRADE_TEMPLATE,
        "twse_daily_template": TWSE_DAILY_TEMPLATE,
        "tpex_daily_template": TPEX_DAILY_TEMPLATE,
    }
    kwargs.update({key: value for key, value in optional_values.items() if value})
    return kwargs


TRADE_DATES = select_calendar_trade_dates(CALENDAR_FILE, TRADE_START_DATE, TRADE_END_DATE)
daily_build_rows: list[dict[str, Any]] = []
daily_pair_records: list[dict[str, Any]] = []

for trade_date in TRADE_DATES:
    kwargs = daily_builder_kwargs(trade_date)
    config_path = Path(kwargs["output"])
    target_path = Path(kwargs["target_output"])
    status = "existing"
    ldate = None
    pair_count = None
    target_count = None

    if REBUILD_DAILY_CONFIGS or not config_path.exists() or not target_path.exists():
        try:
            result = build_arbitrage_config_from_date(trade_date, **kwargs)
            status = "generated"
            ldate = result.ldate
            pair_count = result.pair_count
            target_count = result.target_count
        except Exception as exc:
            daily_build_rows.append({"trade_date": trade_date, "status": "error", "error": repr(exc), "config_path": str(config_path), "target_path": str(target_path)})
            if RAISE_ON_DAILY_CONFIG_ERROR:
                raise
            continue

    app_config = load_config(config_path, replay_date_override=trade_date)
    pairs_for_date = list(app_config.pairs)
    if PAIR_NAME_FILTER:
        allowed = set(PAIR_NAME_FILTER)
        pairs_for_date = [pair for pair in pairs_for_date if pair.name in allowed]
    daily_build_rows.append({
        "trade_date": trade_date,
        "ldate": ldate,
        "status": status,
        "config_path": str(config_path),
        "target_path": str(target_path),
        "pairs": pair_count if pair_count is not None else len(pairs_for_date),
        "targets": target_count,
    })
    for pair in pairs_for_date:
        daily_pair_records.append({"trade_date": trade_date, "run_key": pair_run_key(trade_date, pair.name), "pair": pair, "daily_config_path": config_path})

if MAX_PAIRS is not None:
    daily_pair_records = daily_pair_records[:MAX_PAIRS]

selected_pairs = [record["pair"] for record in daily_pair_records]
daily_pair_lookup = {record["run_key"]: record for record in daily_pair_records}

daily_config_build_df = pd.DataFrame(daily_build_rows)
pair_universe_df = pd.DataFrame([
    {
        "trade_date": record["trade_date"],
        "run_key": record["run_key"],
        "pair_name": record["pair"].name,
        "spot_symbol": record["pair"].spot_symbol,
        "future_symbol": record["pair"].future_symbol,
        "entry_threshold_pct": record["pair"].entry_threshold_pct,
        "min_effective_tick_multiple": record["pair"].min_effective_tick_multiple,
        "spot_order_qty": record["pair"].spot_order_qty,
        "future_order_qty": record["pair"].future_order_qty,
        "future_pnl_multiplier": record["pair"].future_pnl_multiplier,
        "daily_config_path": str(record["daily_config_path"]),
    }
    for record in daily_pair_records
])

daily_config_build_df.to_csv(OUTPUT_DIR / "daily_config_build_status.csv", index=False, encoding="utf-8-sig")
pair_universe_df.to_csv(OUTPUT_DIR / "daily_pair_universe.csv", index=False, encoding="utf-8-sig")

print(f"trade_dates={TRADE_DATES}")
print(f"selected_daily_pairs={len(daily_pair_records)} across {len(TRADE_DATES)} trade dates")
display(daily_config_build_df)
display(pair_universe_df)


FileNotFoundError: [Errno 2] No such file or directory: 'Z:\\TWSE\\瘥\ue432\ue8d9?\uf560\uef94?\x80瘜\ueed420260521.csv'

## Generate Daily HBT Event Data

本 cell 先調用 `scripts/tw_stock_data_to_npz.py`，把每天每個 selected pair 的現貨與股票期貨 top-5 data 轉成 HBT event `.npz`。  
調用 scripts：`scripts/tw_stock_data_to_npz.py` 的 `convert_tw_stock_to_npz`、`convert_tw_stock_future_to_npz`、`default_output_path`。


In [3]:
# 本 cell：逐日逐 pair 產生 spot/future HBT npz event data。
# 調用 scripts:
# - scripts/tw_stock_data_to_npz.py: default_output_path
# - scripts/tw_stock_data_to_npz.py: convert_tw_stock_to_npz
# - scripts/tw_stock_data_to_npz.py: convert_tw_stock_future_to_npz


def expected_event_path(symbol: str, source_kind: str, trade_date: str) -> Path:
    return default_output_path(
        ROOT,
        symbol,
        trade_date,
        trade_date,
        SESSION_START_TIME,
        SESSION_END_TIME,
        source_kind=source_kind,
    )


def ensure_spot_events(symbol: str, trade_date: str) -> tuple[Path | None, str, str | None]:
    output = expected_event_path(symbol, "stock", trade_date)
    if output.exists() and not REBUILD_EVENT_DATA:
        return output, "existing", None
    if not CONVERT_MISSING_EVENT_DATA and not REBUILD_EVENT_DATA:
        return None, "missing", f"missing spot npz: {output}"
    try:
        path, _ = convert_tw_stock_to_npz(
            symbol=symbol,
            start_date=trade_date,
            end_date=trade_date,
            start_time=SESSION_START_TIME,
            end_time=SESSION_END_TIME,
            output=output,
            workspace_root=ROOT,
            data_api=True,
            daily_parquet=False,
            levels=5,
            qa_sample_rows=CONVERSION_QA_SAMPLE_ROWS,
        )
        return path, "generated", None
    except Exception as exc:
        return None, "error", repr(exc)


def ensure_future_events(symbol: str, trade_date: str) -> tuple[Path | None, str, str | None]:
    output = expected_event_path(symbol, "stock_future", trade_date)
    if output.exists() and not REBUILD_EVENT_DATA:
        return output, "existing", None
    if not CONVERT_MISSING_EVENT_DATA and not REBUILD_EVENT_DATA:
        return None, "missing", f"missing future npz: {output}"
    try:
        path, _ = convert_tw_stock_future_to_npz(
            symbol=symbol,
            start_date=trade_date,
            end_date=trade_date,
            start_time=SESSION_START_TIME,
            end_time=SESSION_END_TIME,
            output=output,
            workspace_root=ROOT,
            path_config=ROOT / "path.toml",
            levels=5,
            qa_sample_rows=CONVERSION_QA_SAMPLE_ROWS,
        )
        return path, "generated", None
    except Exception as exc:
        return None, "error", repr(exc)


event_cache: dict[tuple[str, str, str], tuple[Path | None, str, str | None]] = {}
conversion_rows = []
pair_event_paths: dict[str, dict[str, Path]] = {}

for record in daily_pair_records:
    trade_date = record["trade_date"]
    run_key = record["run_key"]
    pair = record["pair"]

    spot_cache_key = (trade_date, "stock", pair.spot_symbol)
    future_cache_key = (trade_date, "stock_future", pair.future_symbol)
    if spot_cache_key not in event_cache:
        event_cache[spot_cache_key] = ensure_spot_events(pair.spot_symbol, trade_date)
    if future_cache_key not in event_cache:
        event_cache[future_cache_key] = ensure_future_events(pair.future_symbol, trade_date)

    spot_path, spot_status, spot_error = event_cache[spot_cache_key]
    future_path, future_status, future_error = event_cache[future_cache_key]
    ok = spot_path is not None and future_path is not None
    if ok:
        pair_event_paths[run_key] = {"spot": spot_path, "future": future_path}
    conversion_rows.append(
        {
            "trade_date": trade_date,
            "run_key": run_key,
            "pair_name": pair.name,
            "spot_symbol": pair.spot_symbol,
            "future_symbol": pair.future_symbol,
            "spot_status": spot_status,
            "future_status": future_status,
            "spot_path": None if spot_path is None else str(spot_path),
            "future_path": None if future_path is None else str(future_path),
            "ok": ok,
            "spot_error": spot_error,
            "future_error": future_error,
        }
    )

conversion_df = pd.DataFrame(conversion_rows)
conversion_df.to_csv(OUTPUT_DIR / "conversion_status.csv", index=False, encoding="utf-8-sig")
print(f"event_data_ready={len(pair_event_paths)} / {len(daily_pair_records)} daily pairs")
display(conversion_df)


NameError: name 'daily_pair_records' is not defined

## HBT Settings Audit

本 cell 用 `scripts/tw_stock_hftbacktest.py` 寫入並稽核每日每個 asset 的 HBT 設定與 event data 摘要。  
調用 scripts：`scripts/tw_stock_hftbacktest.py` 的 `BacktestConfig`、`event_summary`；`future_spot/arbitrage/hbt_backtest.py` 的 `infer_hbt_asset_tick_size`。


In [ ]:
# 本 cell：建立每日 spot/future asset 的 HBT BacktestConfig audit DataFrame。
# 調用 scripts:
# - scripts/tw_stock_hftbacktest.py: BacktestConfig
# - scripts/tw_stock_hftbacktest.py: event_summary
# - future_spot/arbitrage/hbt_backtest.py: infer_hbt_asset_tick_size


def ms_to_ns(value: float) -> int:
    return int(round(value * 1_000_000))


def summarize_asset(trade_date: str, run_key: str, pair: PairConfig, leg: str, data_path: Path) -> dict[str, Any]:
    instrument = "stock" if leg == "spot" else "future"
    contract_size = 1000.0 if leg == "spot" else float(pair.future_pnl_multiplier)
    tick_size = infer_hbt_asset_tick_size(data_path, instrument)
    hbt_config = BacktestConfig(
        data=data_path,
        contract_size=contract_size,
        tick_size=tick_size,
        lot_size=1.0,
        maker_fee=0.0,
        taker_fee=0.0,
        order_latency_ns=ms_to_ns(ORDER_LATENCY_MS),
        queue_model=QUEUE_MODEL,
    )
    data = np.load(data_path)["data"]
    summary = event_summary(data)
    return {
        "trade_date": trade_date,
        "run_key": run_key,
        "pair_name": pair.name,
        "leg": leg,
        "symbol": pair.spot_symbol if leg == "spot" else pair.future_symbol,
        "data": str(hbt_config.data),
        "contract_size": hbt_config.contract_size,
        "tick_size": hbt_config.tick_size,
        "lot_size": hbt_config.lot_size,
        "order_latency_ns": hbt_config.order_latency_ns,
        "queue_model": hbt_config.queue_model,
        "rows": summary["rows"],
        "first_exch_ts": summary["first_exch_ts"],
        "last_exch_ts": summary["last_exch_ts"],
        "min_feed_latency_ns": summary["min_latency_ns"],
        "max_feed_latency_ns": summary["max_latency_ns"],
        "depth_events": summary["depth_events"],
        "trade_events": summary["trade_events"],
    }


settings_rows = []
for record in daily_pair_records:
    run_key = record["run_key"]
    paths = pair_event_paths.get(run_key)
    if not paths:
        continue
    pair = record["pair"]
    trade_date = record["trade_date"]
    settings_rows.append(summarize_asset(trade_date, run_key, pair, "spot", paths["spot"]))
    settings_rows.append(summarize_asset(trade_date, run_key, pair, "future", paths["future"]))

hbt_settings_df = pd.DataFrame(settings_rows)
hbt_settings_df.to_csv(OUTPUT_DIR / "hbt_settings.csv", index=False, encoding="utf-8-sig")
display(hbt_settings_df)


## Notebook Entry Signal

本 cell 將進場信號直接寫在 notebook 裡，方便調整與可視化檢查；runner 仍負責實際成交與風控。  
調用 scripts：本 cell 不調用 script，signal function 直接在 notebook 內定義。


In [ ]:
# 本 cell：在 notebook 內定義進場信號，並可套用到每個 pair 的 market DataFrame。
# 調用 scripts：不直接調用 script；這裡刻意把 entry signal 寫在 notebook 內。


def notebook_entry_signal(row: pd.Series, pair: PairConfig) -> str:
    long_ok = (
        row["long_spot_short_future_pct"] >= pair.entry_threshold_pct
        and row["long_spot_short_future_ticks"] > pair.min_effective_tick_multiple
        and row["spot_ask_size"] >= pair.stock_min_ask_size
        and row["future_bid_size"] >= pair.future_min_bid_size
    )
    if long_ok:
        return Signal.ENTER_LONG_SPOT_SHORT_FUTURE.value

    short_ok = (
        pair.allow_short_spot
        and row["short_spot_long_future_pct"] <= -pair.entry_threshold_pct
        and row["short_spot_long_future_ticks"] > pair.min_effective_tick_multiple
        and row["spot_bid_size"] >= pair.stock_min_bid_size
        and row["future_ask_size"] >= pair.future_min_ask_size
    )
    if short_ok:
        return Signal.ENTER_SHORT_SPOT_LONG_FUTURE.value
    return Signal.HOLD.value


def attach_notebook_entry_signals(market: pd.DataFrame, pair: PairConfig) -> pd.DataFrame:
    if market.empty:
        return market
    result = market.copy()
    result["notebook_entry_signal"] = result.apply(lambda row: notebook_entry_signal(row, pair), axis=1)
    result["notebook_entry_signal_hit"] = result["notebook_entry_signal"].ne(Signal.HOLD.value)
    return result


def entry_signal_output(market: pd.DataFrame) -> pd.DataFrame:
    if market.empty or "notebook_entry_signal_hit" not in market.columns:
        return pd.DataFrame()
    cols = [
        "time", "pair_name", "spot_symbol", "future_symbol", "notebook_entry_signal",
        "spot_bid", "spot_ask", "spot_ask_size", "future_bid", "future_ask", "future_bid_size",
        "long_spot_short_future_pct", "long_spot_short_future_ticks",
        "short_spot_long_future_pct", "short_spot_long_future_ticks",
    ]
    available_cols = [col for col in cols if col in market.columns]
    return market.loc[market["notebook_entry_signal_hit"], available_cols].reset_index(drop=True)


## Run Daily Full-Market Paired HBT

本 cell 執行每天每個 pair 的 two-asset HBT backtest，輸出 trades、summary、market snapshots。  
調用 scripts：`future_spot/arbitrage/hbt_backtest.py` 的 `HbtAssetConfig`、`HbtPairBacktestConfig`、`HbtPairBacktester`。


In [ ]:
# 本 cell：調用 paired HBT runner 跑每日全市場 pair，並附上 notebook entry signal 與 market frame。
# 調用 scripts:
# - future_spot/arbitrage/hbt_backtest.py: HbtAssetConfig
# - future_spot/arbitrage/hbt_backtest.py: HbtPairBacktestConfig
# - future_spot/arbitrage/hbt_backtest.py: HbtPairBacktester


def with_time_columns(df: pd.DataFrame, timestamp_col: str = "timestamp") -> pd.DataFrame:
    if df.empty or timestamp_col not in df.columns:
        return df
    result = df.copy()
    result["time"] = pd.to_datetime(result[timestamp_col], unit="ns", utc=True).dt.tz_convert("Asia/Taipei")
    return result


def add_execution_latency_columns(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    result = df.copy()
    if "signal_timestamp" in result.columns:
        signal_ts = pd.to_numeric(result["signal_timestamp"], errors="coerce")
        for col in ("first_exch_timestamp", "first_local_timestamp", "second_exch_timestamp", "second_local_timestamp", "completion_timestamp"):
            if col in result.columns:
                result[f"{col}_from_signal_ms"] = (pd.to_numeric(result[col], errors="coerce") - signal_ts) / 1_000_000
    if {"first_exch_timestamp", "second_exch_timestamp"}.issubset(result.columns):
        result["first_to_second_exch_ms"] = (
            pd.to_numeric(result["second_exch_timestamp"], errors="coerce")
            - pd.to_numeric(result["first_exch_timestamp"], errors="coerce")
        ) / 1_000_000
    return result


def add_run_columns(df: pd.DataFrame, trade_date: str, run_key: str) -> pd.DataFrame:
    if df.empty:
        return df
    result = df.copy()
    result.insert(0, "run_key", run_key)
    result.insert(0, "trade_date", trade_date)
    return result


def pair_with_overrides(pair: PairConfig) -> PairConfig:
    return replace(pair, **PAIR_OVERRIDES) if PAIR_OVERRIDES else pair


def build_pair_hbt_config(pair: PairConfig, paths: dict[str, Path]) -> HbtPairBacktestConfig:
    return HbtPairBacktestConfig(
        pair=pair_with_overrides(pair),
        spot=HbtAssetConfig(
            symbol=pair.spot_symbol,
            data=paths["spot"],
            instrument="stock",
            contract_size=1000.0,
            order_entry_latency_ns=ms_to_ns(ORDER_LATENCY_MS),
            order_response_latency_ns=ms_to_ns(RESPONSE_LATENCY_MS),
            feed_latency_offset_ns=ms_to_ns(FEED_LATENCY_OFFSET_MS),
            queue_model=QUEUE_MODEL,
        ),
        future=HbtAssetConfig(
            symbol=pair.future_symbol,
            data=paths["future"],
            instrument="future",
            contract_size=float(pair.future_pnl_multiplier),
            order_entry_latency_ns=ms_to_ns(ORDER_LATENCY_MS),
            order_response_latency_ns=ms_to_ns(RESPONSE_LATENCY_MS),
            feed_latency_offset_ns=ms_to_ns(FEED_LATENCY_OFFSET_MS),
            queue_model=QUEUE_MODEL,
        ),
        first_leg=FIRST_LEG,
        step_ns=ms_to_ns(STEP_MS),
        response_timeout_ns=ms_to_ns(RESPONSE_TIMEOUT_MS),
        second_leg_delay_ns=ms_to_ns(SECOND_LEG_DELAY_MS),
        max_steps=MAX_STEPS,
        max_trades=MAX_TRADES_PER_PAIR,
        flatten_on_second_leg_failure=FLATTEN_ON_SECOND_LEG_FAILURE,
        second_leg_profit_check=SECOND_LEG_PROFIT_CHECK,
        record_market_every_steps=RECORD_MARKET_EVERY_STEPS,
    )


pair_results: dict[str, dict[str, pd.DataFrame]] = {}
summary_frames = []
trade_frames = []
market_frames = []
run_errors = []

for record in daily_pair_records:
    trade_date = record["trade_date"]
    run_key = record["run_key"]
    pair = record["pair"]
    paths = pair_event_paths.get(run_key)
    if not paths:
        run_errors.append({"trade_date": trade_date, "run_key": run_key, "pair_name": pair.name, "error": "missing converted event data"})
        continue
    try:
        run_config = build_pair_hbt_config(pair, paths)
        backtester = HbtPairBacktester(run_config)
        trades, summary = backtester.run()
        market = backtester.market_frame()

        pair_for_signal = run_config.pair
        trades = add_run_columns(add_execution_latency_columns(with_time_columns(trades)), trade_date, run_key)
        market = add_run_columns(attach_notebook_entry_signals(with_time_columns(market), pair_for_signal), trade_date, run_key)
        summary = add_run_columns(summary, trade_date, run_key)

        pair_results[run_key] = {"trades": trades, "summary": summary, "market": market}
        summary_frames.append(summary)
        if not trades.empty:
            trade_frames.append(trades)
        if not market.empty:
            market_frames.append(market)
    except Exception as exc:
        run_errors.append({"trade_date": trade_date, "run_key": run_key, "pair_name": pair.name, "error": repr(exc)})

summary_df = pd.concat(summary_frames, ignore_index=True) if summary_frames else pd.DataFrame()
trades_all = pd.concat(trade_frames, ignore_index=True) if trade_frames else pd.DataFrame()
market_all = pd.concat(market_frames, ignore_index=True) if market_frames else pd.DataFrame()
run_errors_df = pd.DataFrame(run_errors)

summary_df.to_csv(OUTPUT_DIR / "summary_all_daily_pairs.csv", index=False, encoding="utf-8-sig")
trades_all.to_csv(OUTPUT_DIR / "trades_all_daily_pairs.csv", index=False, encoding="utf-8-sig")
market_all.to_csv(OUTPUT_DIR / "market_all_daily_pairs.csv", index=False, encoding="utf-8-sig")
run_errors_df.to_csv(OUTPUT_DIR / "run_errors.csv", index=False, encoding="utf-8-sig")

print(f"completed_daily_pairs={len(pair_results)} errors={len(run_errors_df)}")
display(summary_df)
if not run_errors_df.empty:
    display(run_errors_df)


## Per-Symbol Daily Entry / Exit DataFrames

本 cell 將每個 daily pair 的 notebook entry signal 與 HBT execution rows 合併成各自的 DataFrame，存入 `entry_exit_by_pair` dictionary，key 為 `trade_date::pair_name`。  
調用 scripts：不直接調用新的 script；使用前面 cells 產生的 `market_all` / `trades_all`。


In [ ]:
# 本 cell：建立每個 daily pair 的進出場 DataFrame output。
# 調用 scripts：不直接調用 script；使用本 notebook 的 signal function 與 HBT output DataFrame。

DISPLAY_ALL_PAIR_ENTRY_EXIT = True


def build_pair_entry_exit_df(run_key: str) -> pd.DataFrame:
    market = pair_results.get(run_key, {}).get("market", pd.DataFrame())
    trades = pair_results.get(run_key, {}).get("trades", pd.DataFrame())

    signal_df = entry_signal_output(market)
    if not signal_df.empty:
        signal_df = signal_df.copy()
        signal_df["row_type"] = "notebook_entry_signal"
        signal_df["status"] = pd.NA
        signal_df["realized_pnl"] = pd.NA

    if not trades.empty:
        trade_cols = [
            "trade_date", "run_key", "time", "pair_name", "spot_symbol", "future_symbol", "signal", "status", "failure_reason",
            "spot_bid", "spot_ask", "future_bid", "future_ask",
            "long_spot_short_future_pct", "long_spot_short_future_ticks",
            "first_leg", "first_side", "first_requested_price", "first_exec_price", "first_exec_qty",
            "second_leg", "second_side", "second_requested_price", "second_exec_price", "second_exec_qty",
            "first_to_second_exch_ms", "realized_pnl",
        ]
        trade_df = trades[[col for col in trade_cols if col in trades.columns]].copy()
        trade_df["row_type"] = np.where(trade_df["signal"].eq(Signal.EXIT.value), "exit_execution", "entry_execution")
        trade_df = trade_df.rename(columns={"signal": "notebook_entry_signal"})
    else:
        trade_df = pd.DataFrame()

    combined = pd.concat([signal_df, trade_df], ignore_index=True, sort=False)
    if not combined.empty and "time" in combined.columns:
        combined = combined.sort_values("time").reset_index(drop=True)
    return combined


entry_exit_by_pair = {run_key: build_pair_entry_exit_df(run_key) for run_key in pair_results}
entry_exit_all = pd.concat(entry_exit_by_pair.values(), ignore_index=True, sort=False) if entry_exit_by_pair else pd.DataFrame()
entry_exit_all.to_csv(OUTPUT_DIR / "entry_exit_all_daily_pairs.csv", index=False, encoding="utf-8-sig")

entry_exit_index_df = pd.DataFrame([
    {
        "trade_date": daily_pair_lookup.get(run_key, {}).get("trade_date"),
        "run_key": run_key,
        "pair_name": daily_pair_lookup.get(run_key, {}).get("pair").name if run_key in daily_pair_lookup else run_key,
        "rows": len(df),
        "entry_signal_rows": int(df["row_type"].eq("notebook_entry_signal").sum()) if "row_type" in df else 0,
        "entry_execution_rows": int(df["row_type"].eq("entry_execution").sum()) if "row_type" in df else 0,
        "exit_execution_rows": int(df["row_type"].eq("exit_execution").sum()) if "row_type" in df else 0,
    }
    for run_key, df in entry_exit_by_pair.items()
])
entry_exit_index_df.to_csv(OUTPUT_DIR / "entry_exit_index.csv", index=False, encoding="utf-8-sig")

print(f"entry_exit_daily_pairs={len(entry_exit_by_pair)} rows={len(entry_exit_all)}")
display(entry_exit_index_df)

if DISPLAY_ALL_PAIR_ENTRY_EXIT:
    for run_key, df in entry_exit_by_pair.items():
        print(f"run_key={run_key} rows={len(df)}")
        display(df)
else:
    display(entry_exit_all)


## Full-Market Summary Visualization

本 cell 視覺化每日全市場 filled pair 數、second-leg failure 數與 ending quantity，並輸出 signal / execution count table。  
調用 scripts：不直接調用新的 script；使用前面 cells 產生的 summary/trades/signal DataFrame。


In [ ]:
# 本 cell：畫出每日全市場 summary 圖表。
# 調用 scripts：不直接調用 script；使用 summary_df / entry_exit_all。

if summary_df.empty:
    print("No summary rows to plot.")
else:
    plot_df = summary_df.sort_values(["filled_pairs", "second_leg_failures", "final_quantity"], ascending=False).head(30).copy()
    plot_df["plot_label"] = plot_df["trade_date"].astype(str) + " " + plot_df["pair_name"].astype(str)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].bar(plot_df["plot_label"], plot_df["filled_pairs"])
    axes[0].set_title("filled pairs")
    axes[0].tick_params(axis="x", rotation=90)
    axes[0].grid(True, axis="y", alpha=0.25)

    axes[1].bar(plot_df["plot_label"], plot_df["second_leg_failures"], color="tab:red")
    axes[1].set_title("second-leg failures")
    axes[1].tick_params(axis="x", rotation=90)
    axes[1].grid(True, axis="y", alpha=0.25)

    axes[2].bar(plot_df["plot_label"], plot_df["final_quantity"], color="tab:purple")
    axes[2].set_title("ending pair quantity")
    axes[2].tick_params(axis="x", rotation=90)
    axes[2].grid(True, axis="y", alpha=0.25)
    plt.tight_layout()
    plt.show()

if not entry_exit_all.empty:
    signal_counts = entry_exit_all.groupby(["trade_date", "pair_name", "row_type"], dropna=False).size().unstack(fill_value=0)
    display(signal_counts.sort_index())


## Selected Daily Pair Visual Inspection

本 cell 對單一 daily pair 畫 basis path、BBO path、entry/execution points 與 leg timing。  
調用 scripts：不直接調用新的 script；使用 `pair_results` 內的 DataFrame。


In [ ]:
# 本 cell：可視化單一 daily pair；可手動修改 SELECTED_RUN_KEY 指定日期與 pair。
# 調用 scripts：不直接調用 script；使用 pair_results。

SELECTED_RUN_KEY = next(iter(entry_exit_by_pair), None) or (summary_df["run_key"].iloc[0] if not summary_df.empty else None)
# Example override:
# SELECTED_RUN_KEY = "2026-05-21::1301_CFFG6"

if SELECTED_RUN_KEY is None or SELECTED_RUN_KEY not in pair_results:
    print("No selected daily pair is available.")
else:
    selected_market = pair_results[SELECTED_RUN_KEY]["market"]
    selected_trades = pair_results[SELECTED_RUN_KEY]["trades"]
    selected_pair = daily_pair_lookup[SELECTED_RUN_KEY]["pair"]

    if selected_market.empty:
        print(f"{SELECTED_RUN_KEY}: no market snapshots")
    else:
        fig, axes = plt.subplots(2, 1, figsize=(15, 9), sharex=True)
        axes[0].plot(selected_market["time"], selected_market["long_spot_short_future_pct"] * 10_000, label="LS/SF basis bps")
        axes[0].axhline(selected_pair.entry_threshold_pct * 10_000, color="tab:red", linestyle="--", label="entry threshold")
        signal_hits = selected_market[selected_market["notebook_entry_signal_hit"]]
        if not signal_hits.empty:
            axes[0].scatter(signal_hits["time"], signal_hits["long_spot_short_future_pct"] * 10_000, color="tab:orange", s=25, label="notebook entry signal")
        if not selected_trades.empty:
            axes[0].scatter(selected_trades["time"], selected_trades["long_spot_short_future_pct"] * 10_000, color="tab:green", marker="x", s=45, label="execution")
        axes[0].set_title(f"{SELECTED_RUN_KEY} basis and entry/execution points")
        axes[0].set_ylabel("basis bps")
        axes[0].grid(True, alpha=0.25)
        axes[0].legend(loc="best")

        axes[1].plot(selected_market["time"], selected_market["spot_bid"], label="spot bid", color="tab:blue")
        axes[1].plot(selected_market["time"], selected_market["spot_ask"], label="spot ask", color="tab:blue", linestyle="--")
        axes[1].plot(selected_market["time"], selected_market["future_bid"], label="future bid", color="tab:purple")
        axes[1].plot(selected_market["time"], selected_market["future_ask"], label="future ask", color="tab:purple", linestyle="--")
        axes[1].set_title("BBO path")
        axes[1].set_ylabel("price")
        axes[1].grid(True, alpha=0.25)
        axes[1].legend(loc="best")
        plt.tight_layout()
        plt.show()

    if not selected_trades.empty:
        latency_cols = [col for col in ["first_exch_timestamp_from_signal_ms", "second_exch_timestamp_from_signal_ms", "first_to_second_exch_ms"] if col in selected_trades.columns]
        if latency_cols:
            selected_trades[latency_cols].plot(figsize=(12, 4), marker="o", title=f"{SELECTED_RUN_KEY} leg timing")
            plt.ylabel("milliseconds")
            plt.grid(True, alpha=0.25)
            plt.show()
        display(entry_exit_by_pair.get(SELECTED_RUN_KEY, pd.DataFrame()))


## Optional Daily Latency Grid

本 cell 可選擇跑 latency grid，觀察 order latency 對每日成交數、second-leg failure 與 PnL 的影響。  
調用 scripts：`future_spot/arbitrage/hbt_backtest.py` 的 `HbtPairBacktester`。


In [ ]:
# 本 cell：可選的 latency grid；預設關閉，避免每日全市場重跑時間過長。
# 調用 scripts:
# - future_spot/arbitrage/hbt_backtest.py: HbtPairBacktester

RUN_LATENCY_GRID = False
LATENCY_GRID_MS = [0, 1, 5, 10, 25, 50]


def run_pair_latency_case(record: dict[str, Any], paths: dict[str, Path], order_latency_ms: float) -> pd.Series:
    pair = record["pair"]
    cfg = build_pair_hbt_config(pair, paths)
    cfg = replace(
        cfg,
        spot=replace(cfg.spot, order_entry_latency_ns=ms_to_ns(order_latency_ms), order_response_latency_ns=ms_to_ns(RESPONSE_LATENCY_MS)),
        future=replace(cfg.future, order_entry_latency_ns=ms_to_ns(order_latency_ms), order_response_latency_ns=ms_to_ns(RESPONSE_LATENCY_MS)),
        record_market_every_steps=None,
    )
    bt = HbtPairBacktester(cfg)
    _, case_summary = bt.run()
    row = case_summary.iloc[0].copy()
    row["trade_date"] = record["trade_date"]
    row["run_key"] = record["run_key"]
    row["order_latency_ms"] = order_latency_ms
    return row


if RUN_LATENCY_GRID:
    grid_rows = []
    for latency_ms in LATENCY_GRID_MS:
        for record in daily_pair_records:
            paths = pair_event_paths.get(record["run_key"])
            if paths:
                grid_rows.append(run_pair_latency_case(record, paths, latency_ms))
    latency_grid_df = pd.DataFrame(grid_rows)
    latency_grid_df.to_csv(OUTPUT_DIR / "latency_grid.csv", index=False, encoding="utf-8-sig")
    display(latency_grid_df)

    grid_summary = latency_grid_df.groupby("order_latency_ms", as_index=False).agg(
        filled_pairs=("filled_pairs", "sum"),
        second_leg_failures=("second_leg_failures", "sum"),
        realized_pnl=("realized_pnl", "sum"),
    )
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(grid_summary["order_latency_ms"], grid_summary["filled_pairs"], marker="o", label="filled pairs")
    ax.plot(grid_summary["order_latency_ms"], grid_summary["second_leg_failures"], marker="o", label="second-leg failures")
    ax.set_xlabel("order latency ms")
    ax.set_ylabel("count")
    ax.grid(True, alpha=0.25)
    ax.legend(loc="best")
    plt.show()
else:
    print("Set RUN_LATENCY_GRID = True to run daily full-market latency grid.")
